### Generating output_refs.json

In [10]:
from pathlib import Path
import sys
import json
from fable_pyculator import (
    build_notebook_spec,
    derive_output_refs,
    write_output_refs,
)

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for path in (current, *current.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "fable_pyculator").exists():
            return path
    raise RuntimeError("Could not find fable-pyculator repository root")

repo_root = find_repo_root(Path.cwd())
workbook_path = repo_root / Path("tmp/private-workbooks/2024_Open_FABLECalculator.xlsx")
output_ref_path = repo_root / Path("tmp/generated-models/fable-2024/output_refs.json")
contract_path = repo_root / Path("tmp/generated-models/fable-2024/contract.json")
expressions_path = repo_root / Path("tmp/generated-models/fable-2024/expressions.json")
constant_path = repo_root / Path("tmp/generated-models/fable-2024/constants.json")
model_path = repo_root / Path("tmp/generated-models/fable-2024/generated_Fable_2024_model.py")
artifact_dir = repo_root / Path("tmp/generated-models/fable-2024")
artifact_dir.mkdir(parents=True, exist_ok=True)

Build spec

In [11]:
# commented out the headline series for 'TotalResultsWF' in fable_calculator module discovery.py
spec = build_notebook_spec(workbook_path, workbook_id="fable-c-2024")

First inspect the discovered output tables:

In [12]:
for table in spec.output_tables:
    print(table.name, table.sheet, table.range_ref, len(table.cell_refs), "rows")

food_results_diets FOOD K34:Y221 187 rows
food_total_results_diets FOOD K14:AC25 11 rows
production_resultsprod PRODUCTION K59:AA1049 990 rows
production_totalresultsprod PRODUCTION K13:R24 11 rows
production_resultsprodcosts PRODUCTION K34:R45 11 rows
trade_resultstrade TRADE K32:W1022 990 rows
trade_totalresultstrade TRADE K12:O23 11 rows
biodiversity_resultsbiodiv BIODIVERSITY L14:X25 11 rows
biodiversity_resultsprotectedareas BIODIVERSITY L35:S46 11 rows
biodiversity_resultstargetbiodiv BIODIVERSITY L56:AA65 9 rows
biodiversity_resultsagprac BIODIVERSITY L77:Q88 11 rows
biodiversity_resultsbioscore BIODIVERSITY L101:Q110 9 rows
land_resultscroparea LAND K52:V646 594 rows
land_resultsland LAND K16:AK27 11 rows
ghg_resultsghg GHG K17:AM28 11 rows
ghg_gwp_conversion GHG K36:M40 4 rows
water_resultswater WATER K18:O29 11 rows


Confirm if artifact is ready and generate the required output-refs.json

In [13]:
from IPython.display import Markdown, display

missing = [path for path in (workbook_path,) if not path.exists()]
ARTIFACTS_READY = not missing

if missing:
    display(Markdown("Missing local artifact(s):"))
    display([str(path.relative_to(repo_root)) for path in missing])
else:
    display(Markdown("Local 2024 workbook is ready."))

Local 2024 workbook is ready.

In [ ]:
output_refs = derive_output_refs(spec, column_flavour_tags= "OUTPUT-*")
write_output_refs(output_ref_path, output_refs)

('BIODIVERSITY!M36',
 'BIODIVERSITY!M37',
 'BIODIVERSITY!M38',
 'BIODIVERSITY!M39',
 'BIODIVERSITY!M40',
 'BIODIVERSITY!M41',
 'BIODIVERSITY!M42',
 'BIODIVERSITY!M43',
 'BIODIVERSITY!M44',
 'BIODIVERSITY!M45',
 'BIODIVERSITY!M46',
 'BIODIVERSITY!N36',
 'BIODIVERSITY!N37',
 'BIODIVERSITY!N38',
 'BIODIVERSITY!N39',
 'BIODIVERSITY!N40',
 'BIODIVERSITY!N41',
 'BIODIVERSITY!N42',
 'BIODIVERSITY!N43',
 'BIODIVERSITY!N44',
 'BIODIVERSITY!N45',
 'BIODIVERSITY!N46',
 'BIODIVERSITY!O36',
 'BIODIVERSITY!O37',
 'BIODIVERSITY!O38',
 'BIODIVERSITY!O39',
 'BIODIVERSITY!O40',
 'BIODIVERSITY!O41',
 'BIODIVERSITY!O42',
 'BIODIVERSITY!O43',
 'BIODIVERSITY!O44',
 'BIODIVERSITY!O45',
 'BIODIVERSITY!O46',
 'BIODIVERSITY!P15',
 'BIODIVERSITY!P16',
 'BIODIVERSITY!P17',
 'BIODIVERSITY!P18',
 'BIODIVERSITY!P19',
 'BIODIVERSITY!P20',
 'BIODIVERSITY!P21',
 'BIODIVERSITY!P22',
 'BIODIVERSITY!P23',
 'BIODIVERSITY!P24',
 'BIODIVERSITY!P25',
 'BIODIVERSITY!P36',
 'BIODIVERSITY!P37',
 'BIODIVERSITY!P38',
 'BIODIVERSIT

Alternatively

In [15]:
output_refs = sorted(
    {
        row[column_index]
        for table in spec.output_tables
        for column_index, tag in enumerate(table.column_flavour_tags)
        if tag is not None and tag.startswith("OUTPUT-")
        for row in table.cell_refs
    }
)

(artifact_dir / "output_refs.json").write_text(
    json.dumps(output_refs, indent=2),
    encoding="utf-8",
)

len(output_refs), output_refs[:10]

(10274,
 ['BIODIVERSITY!M36',
  'BIODIVERSITY!M37',
  'BIODIVERSITY!M38',
  'BIODIVERSITY!M39',
  'BIODIVERSITY!M40',
  'BIODIVERSITY!M41',
  'BIODIVERSITY!M42',
  'BIODIVERSITY!M43',
  'BIODIVERSITY!M44',
  'BIODIVERSITY!M45'])

### Inferring contract.json expression.json and constants.json using modelwright

In [16]:
from modelwright.cli import (
    _infer_contract_payload,
    _generate_payload,
    build_dependency_graph,
)
from modelwright.extraction import *

In [17]:
import typer
def progress(message: str) -> None:
        if verbose:
            typer.echo(message, err=True)

verbose = True
workbook_record = extract_workbook(workbook_path, progress=progress if verbose else None)
progress("extract workbook done")
progress("build dependency graph start")
graph = build_dependency_graph(workbook_record, progress=progress if verbose else None)
progress("build dependency graph done")

load_workbook formulas start
/home/gafar001/projets/fable-pyculator/.venv/lib/python3.12/site-packages/openpyxl/reader/drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)
/home/gafar001/projets/fable-pyculator/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
load_workbook formulas done
load_workbook cached_values start
load_workbook cached_values done
workbook diagnostics start
workbook diagnostics done
sheets extracted count=62
tables start
tables done count=196
named ranges start
named ranges done count=31
sheet cells start index=1/62 populated=737
sheet cells done index=1/62 extracted=23 total=23
sheet cells start index=2/62 populated=977
sheet cells done index=2/62 extracted=359 total=382
sheet cells start index=3/62 populated=772
sheet cells done index=3/62 extracted=553 total=935
sheet cells start index=4/62 populated=17

In [18]:
_infer_contract_payload(
    workbook=workbook_path,
    module_name='generated_fable_2024_model',
    output_refs='',
    output_refs_file=output_ref_path,
    contract=contract_path,
    expressions=expressions_path,
    constants=constant_path,
    verbose=True
)

extract workbook start
load_workbook formulas start
load_workbook formulas done
load_workbook cached_values start
load_workbook cached_values done
workbook diagnostics start
workbook diagnostics done
sheets extracted count=62
tables start
tables done count=196
named ranges start
named ranges done count=31
sheet cells start index=1/62 populated=737
sheet cells done index=1/62 extracted=23 total=23
sheet cells start index=2/62 populated=977
sheet cells done index=2/62 extracted=359 total=382
sheet cells start index=3/62 populated=772
sheet cells done index=3/62 extracted=553 total=935
sheet cells start index=4/62 populated=178
sheet cells done index=4/62 extracted=139 total=1074
sheet cells start index=5/62 populated=56448
sheet cells done index=5/62 extracted=53877 total=54951
sheet cells start index=6/62 populated=36254
sheet cells done index=6/62 extracted=36035 total=90986
sheet cells start index=7/62 populated=16747
sheet cells done index=7/62 extracted=16169 total=107155
sheet cell

{'contract': {'workbook_id': '2024_Open_FABLECalculator.xlsx',
  'module_name': 'generated_fable_2024_model',
  'entrypoint': 'calculate',
  'input_refs': ['4_calc_land!E29',
   '4_calc_land!E30',
   '4_calc_land!E31',
   '4_calc_land!E32',
   '4_calc_land!E33',
   '4_calc_land!E34',
   '4_calc_land!E35',
   '4_calc_land!E36',
   '4_calc_land!E37',
   '4_calc_land!E38',
   '4_calc_land!E39',
   'BIODIVERSITY!L36',
   'BiodivIntactPA_c!H4',
   'BiodivIntactPA_c!I4',
   'BiodivIntactPA_c!H5',
   'BiodivIntactPA_c!I5',
   'BiodivIntactPA_c!H6',
   'BiodivIntactPA_c!I6',
   'BiodivIntactPA_c!H7',
   'BiodivIntactPA_c!I7',
   'BiodivIntactPA_c!H8',
   'BiodivIntactPA_c!I8',
   'BiodivIntactPA_c!H9',
   'BiodivIntactPA_c!I9',
   'BiodivIntactPA_c!H10',
   'BiodivIntactPA_c!I10',
   'BiodivIntactPA_c!H11',
   'BiodivIntactPA_c!I11',
   'BiodivIntactPA_c!H12',
   'BiodivIntactPA_c!I12',
   'BiodivIntactPA_c!H13',
   'BiodivIntactPA_c!I13',
   'BiodivIntactPA_c!H14',
   'BiodivIntactPA_c!I14',


### Generating a new Fable model using modelwright

In [20]:
_generate_payload(
    contract=contract_path,
    expressions=expressions_path,
    constants=constant_path,
    output=model_path,
)

{'contract': {'workbook_id': '2024_Open_FABLECalculator.xlsx',
  'module_name': 'generated_fable_2024_model',
  'entrypoint': 'calculate',
  'input_refs': ['4_calc_land!E29',
   '4_calc_land!E30',
   '4_calc_land!E31',
   '4_calc_land!E32',
   '4_calc_land!E33',
   '4_calc_land!E34',
   '4_calc_land!E35',
   '4_calc_land!E36',
   '4_calc_land!E37',
   '4_calc_land!E38',
   '4_calc_land!E39',
   'BIODIVERSITY!L36',
   'BiodivIntactPA_c!H4',
   'BiodivIntactPA_c!I4',
   'BiodivIntactPA_c!H5',
   'BiodivIntactPA_c!I5',
   'BiodivIntactPA_c!H6',
   'BiodivIntactPA_c!I6',
   'BiodivIntactPA_c!H7',
   'BiodivIntactPA_c!I7',
   'BiodivIntactPA_c!H8',
   'BiodivIntactPA_c!I8',
   'BiodivIntactPA_c!H9',
   'BiodivIntactPA_c!I9',
   'BiodivIntactPA_c!H10',
   'BiodivIntactPA_c!I10',
   'BiodivIntactPA_c!H11',
   'BiodivIntactPA_c!I11',
   'BiodivIntactPA_c!H12',
   'BiodivIntactPA_c!I12',
   'BiodivIntactPA_c!H13',
   'BiodivIntactPA_c!I13',
   'BiodivIntactPA_c!H14',
   'BiodivIntactPA_c!I14',
